[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Implement the **Adam** optimizer from scratch.

### Signature
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00


In [3]:
import torch

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.lr = lr
        self.beta_1, self.beta_2 = betas
        self.t = 0
        self.eps = eps
        self.params = list(params)
        self.m = [torch.zeros_like(p) for p in params]
        self.v = [torch.zeros_like(p) for p in params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i in range(len(self.params)):
                if self.params[i].grad is None:
                    continue
                self.m[i].mul_(self.beta_1).add_((1 - self.beta_1) * self.params[i].grad)
                self.v[i].mul_(self.beta_2).add_((1 - self.beta_2) * self.params[i].grad ** 2)
                m_cap = self.m[i] / (1 - self.beta_1 ** self.t)
                v_cap = self.v[i] / (1 - self.beta_2 ** self.t)
                self.params[i].add_(-self.lr * m_cap / (torch.sqrt(v_cap) + self.eps))

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [29]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

Step 0: loss=12.5974
Step 1: loss=12.3944
Step 2: loss=12.1939
Step 3: loss=11.9960
Step 4: loss=11.8005


In [30]:
# ✅ SUBMIT
from torch_judge import check
check('adam')


🧪 Testing: Adam Optimizer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Parameters change after step (4.0ms)
  ✅ [2/3] Matches torch.optim.Adam (8.8ms)
  ✅ [3/3] zero_grad works (0.7ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (13.5ms total)
  Progress saved. Run status() to see your dashboard.

